In [1]:
import ast
import pandas as pd
import numpy as np

In [2]:
# data_folder = f'{data_folder}/umap25'
data_folder = f'.'

In [3]:
n_clusters = 32
tracks_distances = pd.read_csv(f'{data_folder}/tracks_percentage_distance_n_clusters{n_clusters}.csv')

In [4]:
tracks_distances.head()

,song_id,percentage_distance,labels
0,9,-0.408141,17
1,11,-0.228522,31
2,23,-0.070360,7
3,24,0.173696,23
4,25,0.234491,25


In [5]:
editorial_playlists = pd.read_csv(f'{data_folder}/editorial_playlists_integer_id.csv')
editorial_playlists.title = editorial_playlists.title.apply(lambda x: x.lower())

In [6]:
editorial_playlists.title.unique()

array(['dub essentials', 'emo essentials', 'pop essentials',
       'r&b essentials', 'folk essentials', 'funk essentials',
       'jazz essentials', 'punk essentials', 'rock essentials',
       'soul essentials', 'anime essentials', 'blues essentials',
       'dance essentials', 'disco essentials', 'divas essentials',
       'duets essentials', 'grime essentials', 'house essentials',
       'k-pop essentials', 'metal essentials', 'grunge essentials',
       'lounge essentials', 'reggae essentials', 'techno essentials',
       'trance essentials', 'ambient essentials', 'art pop essentials',
       'country essentials', 'dubstep essentials', 'acoustic essentials',
       'afrobeat essentials', 'boy band essentials',
       'brit pop essentials', 'festival essentials',
       'hardcore essentials', 'new wave essentials',
       'nu metal essentials', 'pop punk essentials',
       'rap rock essentials', 'sl house essentials',
       'ska punk essentials', 'slow jam essentials',
       'tr

In [7]:
selected = ['pop essentials', 'rock essentials', 'classical essentials', 'electronic essentials']

In [8]:
selected_editorial_playlists = editorial_playlists[editorial_playlists.title.isin(selected)][['title', 'song_id']]

In [9]:
selected_editorial_playlists.head(10)

,title,song_id
3,pop essentials,"[3418, 35738, 58468, 33270, 10581, 69789, 3100..."
12,rock essentials,"[56386, 63041, 49424, 75345, 52792, 23988, 285..."
70,classical essentials,"[48000, 12044, 52893, 71138, 43056, 48157, 667..."
71,classical essentials,"[65365, 59695, 12044, 52893, 71138, 43056, 481..."
87,electronic essentials,"[34634, 60012, 6982, 22757, 37145, 2402, 69847..."
88,electronic essentials,"[34634, 60012, 6982, 22757, 37145, 2402, 69847..."


In [10]:
selected_editorial_playlists.song_id = selected_editorial_playlists.song_id.apply(ast.literal_eval)

In [11]:
selected_editorial_playlists.song_id.apply(lambda x: len(x))

3     70
12    81
70    13
71    22
87    27
88    27
Name: song_id, dtype: int64

In [12]:
selected_editorial_playlists = selected_editorial_playlists.explode('song_id')

In [13]:
selected_editorial_playlists.groupby('title')['song_id'].nunique()

title
classical essentials     23
electronic essentials    27
pop essentials           70
rock essentials          81
Name: song_id, dtype: int64

In [14]:
# Some playlists are duplicated
selected_editorial_playlists = selected_editorial_playlists.drop_duplicates()

In [15]:
how_often_in_editorial = selected_editorial_playlists.groupby('song_id').count()

In [16]:
distances_for_playlists = tracks_distances.merge(selected_editorial_playlists, how='left', on='song_id').fillna('none')

/tmp/ipykernel_264276/1329098343.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  distances_for_playlists = tracks_distances.merge(selected_editorial_playlists, how='left', on='song_id').fillna('none')


In [17]:
# Tracks that are in editorial playlists are closer to the cluster centroids
distances_for_playlists[['percentage_distance', 'title']].groupby('title').nunique()

,percentage_distance
title,
classical essentials,15
electronic essentials,10
none,11790
pop essentials,36
rock essentials,55


In [18]:
distances_for_playlists.head()

,song_id,percentage_distance,labels,title
0,9,-0.408141,17,none
1,11,-0.228522,31,electronic essentials
2,23,-0.070360,7,none
3,24,0.173696,23,none
4,25,0.234491,25,none


In [19]:
# Tracks that are in editorial playlists are closer to the cluster centroids
distances_for_playlists[['percentage_distance', 'title']].groupby('title').mean()

,percentage_distance
title,
classical essentials,-0.223696
electronic essentials,0.016150
none,0.007924
pop essentials,-0.136712
rock essentials,-0.073720


In [20]:
# Check co-occurrence of cluster labels and editorial playlists

In [21]:
tracks_in_playlists = distances_for_playlists[['labels', 'title']]
tracks_in_playlists = tracks_in_playlists[tracks_in_playlists.title.isin(selected)]
tracks_in_playlists

,labels,title
1,31,electronic essentials
175,13,pop essentials
198,31,pop essentials
212,8,rock essentials
221,8,rock essentials
...,...,...
11977,13,pop essentials
11978,31,pop essentials
12163,13,pop essentials
12183,15,rock essentials


In [22]:
# out of the 128, only 22 are in the selected playlists, although there are 162 tracks
tracks_in_playlists.labels.nunique()

10

In [23]:
labels_for_playlist = tracks_in_playlists.groupby('title').agg(['unique'])
labels_for_playlist.columns = labels_for_playlist.columns.map('_'.join)

In [24]:
labels_for_playlist.head()

,labels_unique
title,
classical essentials,[10]
electronic essentials,"[31, 25, 6, 16, 13, 8]"
pop essentials,"[13, 31, 20, 25]"
rock essentials,"[8, 13, 1, 15]"


In [25]:
for title_1 in selected:
    for title_2 in selected:
        if title_1 != title_2:
            set_1 = labels_for_playlist.loc[title_1].labels_unique
            set_1 = set(set_1)
            set_2 = labels_for_playlist.loc[title_2].labels_unique
            set_2 = set(set_2)

            jaccard = len(set_1.intersection(set_2)) / len(set_1.union(set_2))
            print(f'{title_1} ---- {title_2}: {jaccard}')

pop essentials ---- rock essentials: 0.14285714285714285
pop essentials ---- classical essentials: 0.0
pop essentials ---- electronic essentials: 0.42857142857142855
rock essentials ---- pop essentials: 0.14285714285714285
rock essentials ---- classical essentials: 0.0
rock essentials ---- electronic essentials: 0.25
classical essentials ---- pop essentials: 0.0
classical essentials ---- rock essentials: 0.0
classical essentials ---- electronic essentials: 0.0
electronic essentials ---- pop essentials: 0.42857142857142855
electronic essentials ---- rock essentials: 0.25
electronic essentials ---- classical essentials: 0.0


In [26]:
# Jaccard similarity between editorial playlists, in terms of cluster labels
# Only restricting to number of clusters above the elbow, i.e., ns_clusters >=32
# Since for 128 all similarities are < 1/3, we can stop there

ns_clusters = [32, 64, 128]
for n_clusters in ns_clusters:
    tracks_distances = pd.read_csv(f'{data_folder}/tracks_percentage_distance_n_clusters{n_clusters}.csv')
    distances_for_playlists = tracks_distances.merge(selected_editorial_playlists, how='left', on='song_id').fillna('none')

    tracks_in_playlists = distances_for_playlists[['labels', 'title']]
    tracks_in_playlists = tracks_in_playlists[tracks_in_playlists.title.isin(selected)]

    labels_for_playlist = tracks_in_playlists.groupby('title').agg(['unique'])
    labels_for_playlist.columns = labels_for_playlist.columns.map('_'.join)
    print(n_clusters)

    pair_of_selected = [(a, b) for idx, a in enumerate(selected) for b in selected[idx + 1:]]
    jaccards = []
    for title_1, title_2 in pair_of_selected:
        set_1 = labels_for_playlist.loc[title_1].labels_unique
        set_1 = set(set_1)
        set_2 = labels_for_playlist.loc[title_2].labels_unique
        set_2 = set(set_2)

        jaccard = len(set_1.intersection(set_2)) / len(set_1.union(set_2))
        jaccards += [jaccard]
        print(f'{title_1} ---- {title_2}: {jaccard}')
    jaccards = np.array(jaccards)
    print(f'{n_clusters} ---- {jaccards.mean()}')



32
pop essentials ---- rock essentials: 0.14285714285714285
pop essentials ---- classical essentials: 0.0
pop essentials ---- electronic essentials: 0.42857142857142855
rock essentials ---- classical essentials: 0.0
rock essentials ---- electronic essentials: 0.25
classical essentials ---- electronic essentials: 0.0
32 ---- 0.1369047619047619
64
pop essentials ---- rock essentials: 0.125
pop essentials ---- classical essentials: 0.0
pop essentials ---- electronic essentials: 0.4
rock essentials ---- classical essentials: 0.0
rock essentials ---- electronic essentials: 0.18181818181818182
classical essentials ---- electronic essentials: 0.0
64 ---- 0.1178030303030303
128
pop essentials ---- rock essentials: 0.17647058823529413
pop essentials ---- classical essentials: 0.0
pop essentials ---- electronic essentials: 0.26666666666666666
rock essentials ---- classical essentials: 0.0
rock essentials ---- electronic essentials: 0.13333333333333333
classical essentials ---- electronic essenti

/tmp/ipykernel_264276/1620312336.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  distances_for_playlists = tracks_distances.merge(selected_editorial_playlists, how='left', on='song_id').fillna('none')
/tmp/ipykernel_264276/1620312336.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  distances_for_playlists = tracks_distances.merge(selected_editorial_playlists, how='left', on='song_id').fillna('none')
/tmp/ipykernel_264276/1620312336.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a fu